In [4]:
import os
import json
import time
import warnings
import numpy as np
import requests
from typing import Optional
from dataclasses import dataclass, field, asdict

warnings.filterwarnings("ignore")

# Biopython — DNA → protein translation
from Bio.Seq import Seq

# HuggingFace transformers — ESM2 embeddings
import torch
from transformers import AutoTokenizer, AutoModel

print("✓ Imports successful")

✓ Imports successful


In [ ]:
def translate_fragments(dna: str, min_length: int = 10) -> list:
    """
    Returns all ORFs from all 6 frames, sorted by length descending.
    For ORFs not starting with M, also adds a trimmed version starting from the first M
    (if the trimmed version is still >= min_length).
    """
    dna = dna.upper().strip()
    rev_comp = str(Seq(dna).reverse_complement())
    results = []

    for strand in (dna, rev_comp):
        for frame in range(3):
            fragment = strand[frame:]
            fragment = fragment[:len(fragment) - len(fragment) % 3]
            protein = str(Seq(fragment).translate())

            for orf in protein.split("*"):
                if len(orf) >= min_length:
                    results.append(orf)
                    # If doesn't start with M, also add M-trimmed version
                    if not orf.startswith("M") and "M" in orf:
                        trimmed = orf[orf.index("M"):]
                        if len(trimmed) >= min_length:
                            results.append(trimmed)
    #sort by length and alphabetically for same length
    results.sort(key=lambda x: (-len(x), x))    
    return results



In [6]:
seqs = ["gcgtgcgatgaatttggccatattaaactgatgaacccgcagcgcagcaccgtgtggtat",
         "atgaacccgcagcgcagcaccgtgtggtatgcgtgcgatgaatttggccatatgaacccgcagcgc", 
         "ataccacacggtgctgcgctgcgggttcatcagtttaatatggccaaattcatcgcacgc",
         "gcgctgcgggttcatatggccaaattcatcgcacgcataccacacggtgctgcgctgcgggttcat"]

In [9]:
for seq in seqs:
    print(f"DNA: {seq}")
    proteins = translate_fragments(seq)
    print(f"Translated ORFs: {proteins}\n")

DNA: gcgtgcgatgaatttggccatattaaactgatgaacccgcagcgcagcaccgtgtggtat
Translated ORFs: ['ACDEFGHIKLMNPQRSTVWY', 'IPHGAALRVHQFNMAKFIAR', 'YHTVLRCGFISLIWPNSSH', 'TDEPAAQHRVV', 'TTRCCAAGSSV', 'MNPQRSTVWY']

DNA: atgaacccgcagcgcagcaccgtgtggtatgcgtgcgatgaatttggccatatgaacccgcagcgc
Translated ORFs: ['ALRVHMAKFIARIPHGAALRVH', 'MNPQRSTVWYACDEFGHMNPQR', 'AAGSYGQIHRTHTTRCCAAGS', 'RCGFIWPNSSHAYHTVLRCGF', 'MAKFIARIPHGAALRVH', 'TRSAAPCGMRAMNLAI', 'EPAAQHRVVCVR']

DNA: ataccacacggtgctgcgctgcgggttcatcagtttaatatggccaaattcatcgcacgc
Translated ORFs: ['ACDEFGHIKLMNPQRSTVWY', 'IPHGAALRVHQFNMAKFIAR', 'YHTVLRCGFISLIWPNSSH', 'TDEPAAQHRVV', 'TTRCCAAGSSV', 'MNPQRSTVWY']

DNA: gcgctgcgggttcatatggccaaattcatcgcacgcataccacacggtgctgcgctgcgggttcat
Translated ORFs: ['ALRVHMAKFIARIPHGAALRVH', 'MNPQRSTVWYACDEFGHMNPQR', 'AAGSYGQIHRTHTTRCCAAGS', 'RCGFIWPNSSHAYHTVLRCGF', 'MAKFIARIPHGAALRVH', 'TRSAAPCGMRAMNLAI', 'EPAAQHRVVCVR']

